# CriticalGraphRAG — Incremento 03 (Modelado de la solución)

**Ciencia de datos aplicada — ITBA — Tercer entregable**

Este notebook documenta el modelado y la evaluación del chatbot de análisis de conflictos basado en Knowledge Graph + RAG, construido a partir del dataset ACLED (Israel, 2023).

**Contenido alineado con la consigna:**

1. Recap del problema y datos.
2. Arquitectura adoptada: justificación y diseño.
3. Pipeline reproducible (carga, embeddings, índice vectorial).
4. Evaluación cuantitativa: subset gold (18) con ground truth + cualitativa (50).
5. Casos representativos.
6. Reflexión crítica y mejoras propuestas.
7. Persistencia: cómo recargar el sistema sin re-entrenar.

In [1]:
import sys
from pathlib import Path

# Permitir importar src/ desde la notebook
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

## 1. Recap del problema y datos

**Problema**: construir un sistema capaz de responder preguntas en lenguaje natural sobre eventos del conflicto en Israel/Palestina, combinando recuperación estructurada (Cypher sobre Knowledge Graph) y semántica (embeddings vectoriales).

**Dataset**: ACLED — Armed Conflict Location & Event Data. Filtro aplicado (`config/dataset_filter.yaml`):
- `country: ['Israel']`
- `year:    [2023]`
- Resultado: **4259 eventos** documentados en los 6 distritos administrativos de Israel.

El detalle del EDA se documentó en `notebooks/02_eda_acled_israel.ipynb` (Incremento 02).

## 2. Arquitectura adoptada

**Decisión: RAG sobre Knowledge Graph** (no RAG plano sobre texto). Justificación:

1. ACLED es naturalmente relacional (Event ↔ Actor ↔ Location ↔ EventType ↔ Source). Un KG preserva esas relaciones y permite preguntas multi-hop.
2. Muchas preguntas son factuales (conteos, agregaciones, rankings) — un índice vectorial solo no responde "¿cuántas fatalidades hubo en X mes?".
3. El embedding semántico se reserva para preguntas descriptivas (`Event.notes`), evitando inflar el índice con texto trivial.

**Componentes**:

```
Pregunta del usuario
        │
        ▼
   Agente LLM (Gemini 2.0 Flash, LangGraph)
        │
   ┌────┴──────────────┐
   │      Decide       │
   ▼                   ▼
cypher_query     similarity_search
(query catálogo)   (búsqueda vectorial
                    sobre Event.notes)
        │                   │
        └──────┬────────────┘
               ▼
        Neo4j Knowledge Graph
```

**Tools del agente:**

| Tool | Qué hace | Cuándo se usa | Estado |
|------|----------|---------------|--------|
| `cypher_query` | Ejecuta una query Cypher predefinida de `config/cypher_library.yaml`. El planner LLM elige el `query_id` y completa los `parameters`. | Preguntas factuales / agregación / ranking / comparaciones temporales | ✅ Implementada (14 queries en el catálogo) |
| `similarity_search` | Embebe la pregunta y recupera top-k Events más similares sobre el índice vectorial. | Preguntas descriptivas, búsqueda por contenido | ✅ Implementada |
| `text2cypher` | LLM genera Cypher dinámico cuando ninguna query del library encaja. | Fallback general | ⚠️ Scaffolding (mejora futura — ver sección 6) |

In [2]:
# Mostrar el catálogo actual de cypher_query
from src.tools.cypher_query import CypherLibrary

lib = CypherLibrary(str(ROOT / 'config' / 'cypher_library.yaml'))
catalog = pd.DataFrame([
    {'id': q.id, 'description': q.description, 'params': ', '.join(p['name'] for p in q.parameters)}
    for q in lib.all()
])
print(f'Total queries en el catálogo: {len(catalog)}')
catalog

Total queries en el catálogo: 14


,id,description,params
0,count_events_by_country_month,Cuenta eventos en un país y un mes YYYY-MM específicos.,"country, month_id"
1,sum_fatalities_by_country_year,Suma las fatalidades registradas en un país durante un año.,"country, year"
2,sum_fatalities_by_admin1_year,"Suma de fatalidades filtrando por admin1 (región: Gaza, West Bank, etc.) y año.","admin1, year"
3,top_actors_by_event_count,Top N actores con más apariciones en eventos dentro de una región y año.,"admin1, year, top_n"
4,events_by_event_type_period,Cuenta eventos de un tipo específico (event_type primario o sub_event_type) en un rango de fechas y región.,"event_type_name, start_date, end_date, admin1"
5,actors_co_occurring_with,Actores que aparecieron en los mismos eventos que un actor dado.,"actor_name, admin1, year, top_n"
6,sum_fatalities_by_country_date,Suma de fatalidades en un país en una fecha exacta (YYYY-MM-DD).,"country, event_date"
7,count_events_by_admin1_year,Cuenta eventos en un admin1 (distrito/región) durante un año.,"admin1, year"
8,peak_month_events_by_country_year,Mes con más eventos para un país y año dados (top-1).,"country, year"
9,count_events_by_country_month_list,Cuenta eventos en un país para una lista de meses YYYY-MM.,"country, month_ids"


## 3. Pipeline reproducible

El pipeline está partido en 6 scripts independientes, cada uno persiste su salida para poder reiniciar desde cualquier checkpoint sin recomputar.

```bash
poetry run python scripts/01_download.py        # CSV crudo de ACLED
poetry run python scripts/02_explore.py         # EDA (notebook: 02_eda_acled_israel.ipynb)
poetry run python scripts/03_build_graph.py     # CSV → parquets nodes/rels
poetry run python scripts/04_embeddings.py      # Event.notes → vectores (Gemini)
poetry run python scripts/05_load_neo4j.py --mode destructive   # carga Neo4j
poetry run python scripts/06_evaluate.py        # ground truth + evaluación
```

Las salidas intermedias quedan en `data/graph/` (parquets) y `data/event_embeddings.parquet`. El grafo se reconstruye en cualquier instancia Neo4j ejecutando solo `05`.

In [3]:
# Conteos del KG cargado (sanity check)
from src.neo4j_conn import get_driver

with get_driver() as drv, drv.session() as session:
    nodes_total = session.run('MATCH (n) RETURN count(n) AS n').single()['n']
    rels_total = session.run('MATCH ()-[r]->() RETURN count(r) AS n').single()['n']
    by_label = session.run("""
        MATCH (n)
        WITH labels(n)[0] AS label, count(*) AS count
        RETURN label, count ORDER BY count DESC
    """).data()
    by_rel = session.run("""
        MATCH ()-[r]->()
        WITH type(r) AS rtype, count(*) AS count
        RETURN rtype, count ORDER BY count DESC
    """).data()
    vec_idx = session.run("SHOW INDEXES YIELD name, type WHERE type='VECTOR' RETURN name").data()

print(f'Total nodos:        {nodes_total}')
print(f'Total relaciones:   {rels_total}')
print(f'Vector index:       {vec_idx}')
print('\nNodos por label:')
for row in by_label:
    print(f"  {row['label']:18s} {row['count']:>6}")
print('\nRelaciones por tipo:')
for row in by_rel:
    print(f"  {row['rtype']:18s} {row['count']:>6}")

Total nodos:        5134
Total relaciones:   36329
Vector index:       [{'name': 'event_notes_embedding'}]

Nodos por label:
  Event                4259
  Location              418
  Source                272
  Actor                 139
  EventType              25
  Month                  12
  ActorType               6
  DisorderType            3

Relaciones por tipo:
  REPORTED_BY          9748
  INVOLVED_IN          9482
  OF_DISORDER          4263
  OF_TYPE              4259
  IN_MONTH             4259
  AT_LOCATION          4259
  HAS_TYPE               40
  SUBTYPE_OF             19


## 4. Evaluación

### 4.1 Estrategia

Sistemas RAG no admiten una métrica única de exactitud como un clasificador. Adoptamos un esquema en dos niveles:

- **Subset gold (18 preguntas)**: ground truth objetivo derivado directamente del grafo (queries Cypher de referencia en `tests/gold_cypher.yaml`). Estratificado por tipo de pregunta:
  - Factual simple (4), Agregación (3), Comparación temporal (2),
  - Relaciones entre actores (3), Geoespacial (2), Tipo de evento (2),
  - Ranking (1), Negativa/Ausencia (1).
- **Análisis cualitativo (50 preguntas del CSV original)**: ejecutamos el agente y analizamos qué tool eligió, latencia y plausibilidad. No tienen ground truth automático.

### 4.2 Métricas

- **Numeric match (±5%)** para respuestas numéricas (conteos, sumas, fatalidades). Tolerancia para absorber diferencias de phrasing.
- **Set overlap (Jaccard sobre tokens)** para listas y rankings.
- **Substring Sí/No** para preguntas de ausencia.

### 4.3 Ground truth (subset gold)

In [4]:
gold = pd.read_csv(ROOT / 'tests' / 'gold_subset.csv', dtype=str, keep_default_na=False)
print(f'Subset gold: {len(gold)} preguntas')
gold[['id', 'tipo', 'pregunta', 'respuesta_esperada']]

Subset gold: 18 preguntas


,id,tipo,pregunta,respuesta_esperada
0,g01,Factual simple,¿Cuántos eventos registró ACLED en Israel durante octubre de 2023?,768
1,g02,Factual simple,¿Cuántas fatalidades se registraron en Israel el 7 de octubre de 2023?,1569
2,g03,Factual simple,¿Cuántos eventos de tipo 'Protests' (event_type primario) se registraron en Israel durante 2023?,2413
3,g04,Factual simple,¿Cuántos eventos registró ACLED en el distrito de Tel Aviv durante todo el año 2023?,610
4,g05,Agregación / Conteo,¿Cuántas fatalidades totales registró ACLED en Israel durante 2023?,1773
5,g06,Agregación / Conteo,¿Cuántos eventos de tipo 'Explosions/Remote violence' (primario) ocurrieron en Israel entre octubre y diciembre de 2...,819
6,g07,Agregación / Conteo,¿Cuántos eventos registró ACLED en el distrito de HaDarom durante todo 2023?,1024
7,g08,Comparación temporal,¿En qué mes de 2023 se registraron más eventos en Israel?,"month=2023-10, event_count=768"
8,g09,Comparación temporal,¿Cuántos eventos más ocurrieron en Israel en octubre de 2023 respecto a septiembre de 2023?,588
9,g10,Relaciones entre actores,¿Qué actores participaron en eventos junto a las Fuerzas de Defensa de Israel (FDI) en HaZafon durante 2023?,Hezbollah; Protesters (Israel); Civilians (Israel); Hamas Movement; Unidentified Armed Group (Lebanon); Labor Group ...


### 4.4 Ejecutar el agente sobre las preguntas

Para correr (o re-correr) la evaluación end-to-end:

```bash
poetry run python scripts/06_evaluate.py            # ambas fases
poetry run python scripts/06_evaluate.py --only-eval # asumiendo gold ya poblado
```

Requiere cuota disponible de Gemini API (free tier: 100 RPM, 1000 RPD). Las 68 preguntas (18 gold + 50 originales) se procesan en ~5 minutos.

Los resultados quedan en `tests/eval_results.csv`.

In [5]:
eval_path = ROOT / 'tests' / 'eval_results.csv'
if eval_path.exists():
    results = pd.read_csv(eval_path)
    print(f'Eval results cargados: {len(results)} filas ({results["is_gold"].sum()} gold)')
    display(results.head(3))
else:
    print('⚠️  No existe tests/eval_results.csv. Ejecutá:\n   poetry run python scripts/06_evaluate.py')
    results = None

⚠️  No existe tests/eval_results.csv. Ejecutá:
   poetry run python scripts/06_evaluate.py


### 4.5 Métricas — subset gold

In [6]:
if results is not None:
    gold_res = results[results['is_gold']].copy()
    print(f'Accuracy global (subset gold): {gold_res["score"].mean():.3f}')
    print(f'Latencia media (ms):           {gold_res["latency_ms"].mean():.0f}\n')
    print('Accuracy por tipo de pregunta:')
    by_type = gold_res.groupby('tipo').agg(
        n=('id', 'count'),
        accuracy=('score', 'mean'),
        lat_ms=('latency_ms', 'mean'),
    ).round(3)
    display(by_type)

In [7]:
if results is not None:
    print('Tool elegida por el agente (todas las preguntas):')
    print(results['first_tool'].value_counts(dropna=False).to_string())
    print('\nTool elegida por el agente (subset gold):')
    print(gold_res['first_tool'].value_counts(dropna=False).to_string())

## 5. Casos representativos

Top-3 mejores (score alto) y bottom-3 peores (score bajo) del subset gold para análisis cualitativo.

In [8]:
if results is not None:
    print('### TOP 3 — respuestas más alineadas\n')
    top = gold_res.nlargest(3, 'score')[['id', 'tipo', 'pregunta', 'respuesta_esperada', 'respuesta_modelo', 'score']]
    for _, r in top.iterrows():
        print(f"[{r['id']}] {r['tipo']} — score={r['score']:.2f}")
        print(f"  Q: {r['pregunta']}")
        print(f"  Esperado: {r['respuesta_esperada'][:120]}")
        print(f"  Modelo:   {r['respuesta_modelo'][:200]}\n")
    print('\n### BOTTOM 3 — respuestas peores\n')
    bot = gold_res.nsmallest(3, 'score')[['id', 'tipo', 'pregunta', 'respuesta_esperada', 'respuesta_modelo', 'score', 'first_tool']]
    for _, r in bot.iterrows():
        print(f"[{r['id']}] {r['tipo']} — score={r['score']:.2f} (tool={r['first_tool']})")
        print(f"  Q: {r['pregunta']}")
        print(f"  Esperado: {r['respuesta_esperada'][:120]}")
        print(f"  Modelo:   {r['respuesta_modelo'][:200]}\n")

## 6. Reflexión crítica y mejoras propuestas

### 6.1 Fortalezas observadas

- **Separación clara entre estructurado y semántico**: `cypher_query` da respuestas exactas para conteos; `similarity_search` aporta contexto descriptivo. El planner LLM decide bien para preguntas con patrón obvio.
- **Catálogo extensible**: agregar una query nueva al library es solo editar YAML; el planner la descubre por la `description` y `when_to_use` inyectadas en su prompt.
- **Persistencia robusta**: el pipeline persiste cada etapa en parquets/Neo4j, permitiendo reiniciar desde checkpoint.

### 6.2 Limitaciones conocidas

- **Cobertura del catálogo**: el library cubre los patrones del subset gold pero no es exhaustivo. Preguntas que requieran agregaciones nuevas requieren agregar una entrada.
- **`text2cypher` no implementado**: el fallback LLM-to-Cypher quedó como scaffolding. Cuando una pregunta no encaja en el library, el agente debe usar `similarity_search` o admitir que no puede.
- **Dataset acotado a Israel-2023**: muchas preguntas "naturales" sobre el conflicto involucran Gaza y Cisjordania (`country=Palestine` en ACLED), que están fuera del filtro actual.
- **Métrica de set-overlap es ruidosa**: tokens como `event_count`, `=` se cuelan; un comparador más estructurado mejoraría la señal.

### 6.3 Mejoras concretas (priorizadas)

1. **Implementar `text2cypher`** con `GraphCypherQAChain` + validación EXPLAIN + retry. El scaffolding está en `src/tools/text2cypher.py`.
2. **Ampliar el dataset** agregando `country: [Israel, Palestine]` y `year: [2023, 2024]`. Requiere re-ejecutar 01 → 05.
3. **Re-ranker para `similarity_search`**: aplicar un cross-encoder sobre top-50 antes de devolver top-10.
4. **Captura estructurada de tool calls** dentro del `AgentState` (hoy se reconstruye post-hoc).
5. **Validación de parámetros** antes de ejecutar el Cypher (regex format para `YYYY-MM`, enum para `admin1`).
6. **Métricas adicionales**: latencia p95, costo en tokens, distribución de retries por timeout/quota.

### 6.4 Discusión por tipo de pregunta

- **Factual simple / Agregación / Ranking**: el sistema rinde bien cuando hay query directa en el library. Errores aparecen cuando el planner elige parámetros incorrectos (ej: año o admin1 mal interpretado).
- **Comparación temporal**: requieren composición (dos llamadas o aritmética post-hoc). Hoy se resuelve con queries que devuelven ambos valores y el LLM hace la resta.
- **Relaciones entre actores**: dependen del modelado correcto de `INVOLVED_IN.side`. La query `top_opponents_of_actor` filtra explícitamente sides opuestos para distinguir oposición de co-aparición.
- **Negativa / Ausencia**: la tool devuelve un resultado vacío y el LLM debe afirmar "ninguno". Es donde más se nota la importancia del system prompt.

## 7. Persistencia del modelo / sistema

El sistema completo es **reutilizable sin reentrenamiento**:

| Artefacto | Ubicación | Cómo se regenera |
|-----------|-----------|------------------|
| Dataset crudo | `data/acled_israel_2023.csv` | `scripts/01_download.py` |
| Nodos del KG  | `data/graph/nodes/*.parquet` | `scripts/03_build_graph.py` |
| Relaciones    | `data/graph/relationships/*.parquet` | `scripts/03_build_graph.py` |
| Embeddings    | `data/graph/event_embeddings.parquet` | `scripts/04_embeddings.py` |
| Grafo Neo4j   | DB en `bolt://localhost:7687` | `scripts/05_load_neo4j.py --mode destructive` |
| Vector index  | Neo4j (`event_notes_embedding`, dim=1536, cosine) | creado por `05` automáticamente |
| Catálogo de queries | `config/cypher_library.yaml` | editado a mano |
| Ground truth gold   | `tests/gold_subset.csv` + `tests/gold_cypher.yaml` | `scripts/06_evaluate.py --only-gold` |

**Restore en otra máquina**:

```bash
git clone <repo>
cd CriticalGraphRAG
cp .env.example .env  # completar credenciales
docker-compose up -d neo4j
poetry install
# Saltar 01..04 si los parquets están commiteados o se compartieron:
poetry run python scripts/05_load_neo4j.py --mode destructive
# Listo para servir queries:
poetry run python server.py
```

El servidor FastAPI (`server.py`) levanta el endpoint `/chat` que recibe preguntas vía POST y devuelve respuestas del agente.